In [1]:
import pandas as pd
import psycopg2

print("PostgreSQL connection library ready")

PostgreSQL connection library ready


In [2]:
conn = psycopg2.connect(
    host="localhost",
    port=5432,
    database="medicore_healthcare",
    user="postgres",
    password="Ram@1826"
)

print("Database connection successful")

Database connection successful


In [3]:
query = """
SELECT
    appointment_id,
    booking_channel,
    lead_time_days,
    no_show,
    status
FROM fact_appointments;
"""

df = pd.read_sql_query(query, conn)

print("Rows:", len(df))
print("Columns:", df.shape[1])
df.head()

C:\Users\janak\AppData\Local\Temp\ipykernel_1440\4232922189.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, conn)


Rows: 500000
Columns: 5


,appointment_id,booking_channel,lead_time_days,no_show,status
0,A000001,Referral,7,False,Completed
1,A000002,Referral,15,False,Completed
2,A000003,Online,10,False,Completed
3,A000004,Online,2,False,Completed
4,A000005,Phone,5,False,Completed


In [4]:
import scipy
from scipy.stats import chi2_contingency

print("SciPy version:", scipy.__version__)
print("Statistical library ready")

SciPy version: 1.18.0
Statistical library ready


In [5]:
df["lead_time_group"] = pd.cut(
    df["lead_time_days"],
    bins=[-1, 0, 3, 7, 14, 30, float("inf")],
    labels=["Same day", "1–3 days", "4–7 days", "8–14 days", "15–30 days", "31+ days"]
)

pd.crosstab(df["lead_time_group"], df["no_show"])

no_show,False,True
lead_time_group,,
Same day,1374,141
1–3 days,63323,5540
4–7 days,135178,12285
8–14 days,164994,14991
15–30 days,84621,11474
31+ days,5244,835


In [6]:
contingency_table = pd.crosstab(
    df["lead_time_group"],
    df["no_show"]
)

chi2, p_value, dof, expected = chi2_contingency(contingency_table)

print("Chi-square statistic:", chi2)
print("Degrees of freedom:", dof)
print("P-value:", p_value)

Chi-square statistic: 1427.9092063583014
Degrees of freedom: 5
P-value: 1.2337788713269482e-306


In [7]:
import numpy as np

n = contingency_table.to_numpy().sum()
cramers_v = np.sqrt(chi2 / (n * min(contingency_table.shape[0] - 1,
                                    contingency_table.shape[1] - 1)))

print("Cramér's V:", cramers_v)

Cramér's V: 0.05343985790322241


In [8]:
import statsmodels
print("Statsmodels version:", statsmodels.__version__)


Statsmodels version: 0.14.6


In [9]:
import statsmodels.api as sm

X = df[["lead_time_days"]]
X = sm.add_constant(X)

y = df["no_show"].astype(int)

logit_model = sm.Logit(y, X).fit()

print(logit_model.summary())

Optimization terminated successfully.
         Current function value: 0.302821
         Iterations 6
                           Logit Regression Results                           
Dep. Variable:                no_show   No. Observations:               500000
Model:                          Logit   Df Residuals:                   499998
Method:                           MLE   Df Model:                            1
Date:                Sat, 12 Sep 2026   Pseudo R-squ.:                0.003114
Time:                        16:19:19   Log-Likelihood:            -1.5141e+05
converged:                       True   LL-Null:                   -1.5188e+05
Covariance Type:            nonrobust   LLR p-value:                1.014e-207
                     coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------
const             -2.5295      0.009   -285.886      0.000      -2.547      -2.512
lead_time_days   

In [10]:
odds_ratio_1_day = np.exp(logit_model.params["lead_time_days"])
odds_ratio_7_days = np.exp(logit_model.params["lead_time_days"] * 7)

print(f"Odds ratio per 1 day: {odds_ratio_1_day:.4f}")
print(f"Odds ratio per 7 days: {odds_ratio_7_days:.4f}")
print(f"7-day increase in odds: {(odds_ratio_7_days - 1) * 100:.2f}%")

Odds ratio per 1 day: 1.0218
Odds ratio per 7 days: 1.1631
7-day increase in odds: 16.31%


In [11]:
df[[
    "booking_channel",
    "lead_time_days",
    "no_show",
    "status"
]].dtypes

booking_channel      str
lead_time_days     int64
no_show             bool
status               str
dtype: object

In [12]:
query = """
SELECT
    appointment_id,
    appointment_type,
    booking_channel,
    lead_time_days,
    no_show,
    department_id,
    hospital_id
FROM fact_appointments;
"""

df = pd.read_sql_query(query, conn)

print("Rows:", len(df))
print("Columns:", df.shape[1])
df.head()

C:\Users\janak\AppData\Local\Temp\ipykernel_1440\1202583489.py:13: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, conn)


Rows: 500000
Columns: 7


,appointment_id,appointment_type,booking_channel,lead_time_days,no_show,department_id,hospital_id
0,A000001,Consultation,Referral,7,False,D011,H002
1,A000002,Diagnostic,Referral,15,False,D035,H006
2,A000003,Consultation,Online,10,False,D034,H006
3,A000004,Consultation,Online,2,False,D006,H001
4,A000005,Follow-up,Phone,5,False,D018,H004


In [13]:
print("Appointment types:")
print(df["appointment_type"].value_counts())

print("\nBooking channels:")
print(df["booking_channel"].value_counts())

print("\nHospitals:")
print(df["hospital_id"].nunique())

print("\nDepartments:")
print(df["department_id"].nunique())

print("\nMissing values:")
print(df.isnull().sum())

Appointment types:
appointment_type
Consultation    299924
Follow-up       124772
Diagnostic       75304
Name: count, dtype: int64

Booking channels:
booking_channel
Online      199915
Phone       125518
Walk-in     100260
Referral     74307
Name: count, dtype: int64

Hospitals:
8

Departments:
44

Missing values:
appointment_id      0
appointment_type    0
booking_channel     0
lead_time_days      0
no_show             0
department_id       0
hospital_id         0
dtype: int64


In [14]:
appointment_type_table = pd.crosstab(
    df["appointment_type"],
    df["no_show"]
)

print(appointment_type_table)

print("\nNo-show rate by appointment type:")
print(
    df.groupby("appointment_type")["no_show"]
      .mean()
      .mul(100)
      .round(2)
)

no_show            False  True 
appointment_type               
Consultation      273723  26201
Diagnostic         69399   5905
Follow-up         111612  13160

No-show rate by appointment type:
appointment_type
Consultation     8.74
Diagnostic       7.84
Follow-up       10.55
Name: no_show, dtype: float64


In [15]:
from scipy.stats import chi2_contingency

chi2, p_value, dof, expected = chi2_contingency(
    appointment_type_table
)

print("Chi-square statistic:", chi2)
print("Degrees of freedom:", dof)
print("P-value:", p_value)

Chi-square statistic: 509.2106834322027
Degrees of freedom: 2
P-value: 2.6687324083063005e-111


In [16]:
n = appointment_type_table.to_numpy().sum()

cramers_v = np.sqrt(
    chi2 / (
        n * min(
            appointment_type_table.shape[0] - 1,
            appointment_type_table.shape[1] - 1
        )
    )
)

print("Cramér's V:", cramers_v)

Cramér's V: 0.031912714815013864


In [17]:
booking_channel_table = pd.crosstab(
    df["booking_channel"],
    df["no_show"]
)

print(booking_channel_table)

print("\nNo-show rate by booking channel:")
print(
    df.groupby("booking_channel")["no_show"]
      .mean()
      .mul(100)
      .round(2)
)

no_show           False  True 
booking_channel               
Online           179211  20704
Phone            114975  10543
Referral          68758   5549
Walk-in           91790   8470

No-show rate by booking channel:
booking_channel
Online      10.36
Phone        8.40
Referral     7.47
Walk-in      8.45
Name: no_show, dtype: float64


In [18]:
from scipy.stats import chi2_contingency

chi2, p_value, dof, expected = chi2_contingency(
    booking_channel_table
)

print("Chi-square statistic:", chi2)
print("Degrees of freedom:", dof)
print("P-value:", p_value)

Chi-square statistic: 748.9596443954465
Degrees of freedom: 3
P-value: 5.072564902657467e-162


In [19]:
n = booking_channel_table.to_numpy().sum()

cramers_v = np.sqrt(
    chi2 / (
        n * min(
            booking_channel_table.shape[0] - 1,
            booking_channel_table.shape[1] - 1
        )
    )
)

print("Cramér's V:", cramers_v)


Cramér's V: 0.038702962274106266


In [20]:
hospital_table = pd.crosstab(
    df["hospital_id"],
    df["no_show"]
)

print(hospital_table)

print("\nNo-show rate by hospital:")
print(
    df.groupby("hospital_id")["no_show"]
      .mean()
      .mul(100)
      .round(2)
      .sort_values(ascending=False)
)

no_show      False  True 
hospital_id              
H001         60401   6100
H002         67027   6679
H003         47674   4753
H004         56727   5636
H005         79687   8046
H006         59026   5974
H007         43235   4186
H008         40957   3892

No-show rate by hospital:
hospital_id
H006    9.19
H001    9.17
H005    9.17
H003    9.07
H002    9.06
H004    9.04
H007    8.83
H008    8.68
Name: no_show, dtype: float64


In [21]:
from scipy.stats import chi2_contingency

chi2, p_value, dof, expected = chi2_contingency(
    hospital_table
)

print("Chi-square statistic:", chi2)
print("Degrees of freedom:", dof)
print("P-value:", p_value)

Chi-square statistic: 14.770285037986913
Degrees of freedom: 7
P-value: 0.03905937676092897


In [22]:
n = hospital_table.to_numpy().sum()

cramers_v = np.sqrt(
    chi2 / (
        n * min(
            hospital_table.shape[0] - 1,
            hospital_table.shape[1] - 1
        )
    )
)

print("Cramér's V:", cramers_v)

Cramér's V: 0.005435123740631286


In [23]:
department_table = pd.crosstab(
    df["department_id"],
    df["no_show"]
)

print(department_table)

print("\nNo-show rate by department:")
print(
    df.groupby("department_id")["no_show"]
      .mean()
      .mul(100)
      .round(2)
      .sort_values(ascending=False)
)

no_show        False  True 
department_id              
D001           17431   1796
D002           11962   1199
D003            3842    368
D004           10581   1026
D005            9066    949
D006            7519    762
D007            7034    719
D008            9132    851
D009            9740    991
D010            9262    922
D011           17527   1771
D012           14332   1425
D013            9117    900
D014            7691    787
D015            9782    951
D016           10618   1064
D017           10466   1051
D018           14452   1457
D019           10700   1054
D020            9825    951
D021            5163    533
D022            9030    873
D023            7557    768
D024            7608    763
D025           12020   1232
D026            8947    938
D027            7734    749
D028           21946   2230
D029           12383   1193
D030            9049    941
D031            8363    855
D032            7547    750
D033            9768    968
D034            9087

In [24]:
from scipy.stats import chi2_contingency

chi2, p_value, dof, expected = chi2_contingency(
    department_table
)

print("Chi-square statistic:", chi2)
print("Degrees of freedom:", dof)
print("P-value:", p_value)

Chi-square statistic: 39.129012587948225
Degrees of freedom: 43
P-value: 0.6398761713737128


In [27]:
status_query = """
SELECT
    appointment_id,
    status
FROM fact_appointments;
"""

status_df = pd.read_sql_query(status_query, conn)

df = df.merge(
    status_df,
    on="appointment_id",
    how="left"
)

print("Rows:", len(df))
print("\nStatus distribution:")
print(df["status"].value_counts())

print("\nMissing status:", df["status"].isna().sum())

C:\Users\janak\AppData\Local\Temp\ipykernel_1440\826867774.py:8: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  status_df = pd.read_sql_query(status_query, conn)


Rows: 500000

Status distribution:
status
Completed    415965
No-show       45266
Cancelled     38769
Name: count, dtype: int64

Missing status: 0


In [28]:
model_df = df[df["status"].isin(["Completed", "No-show"])].copy()

print("Original appointments:", len(df))
print("Modeling appointments:", len(model_df))

print("\nStatus distribution:")
print(model_df["status"].value_counts())

print("\nNo-show rate in modeling dataset:")
print(f"{model_df['no_show'].mean() * 100:.2f}%")

Original appointments: 500000
Modeling appointments: 461231

Status distribution:
status
Completed    415965
No-show       45266
Name: count, dtype: int64

No-show rate in modeling dataset:
9.81%


In [29]:
print("Missing values:")
print(model_df.isnull().sum())

print("\nData types:")
print(model_df.dtypes)

Missing values:
appointment_id      0
appointment_type    0
booking_channel     0
lead_time_days      0
no_show             0
department_id       0
hospital_id         0
status              0
dtype: int64

Data types:
appointment_id        str
appointment_type      str
booking_channel       str
lead_time_days      int64
no_show              bool
department_id         str
hospital_id           str
status                str
dtype: object


In [30]:
# Prepare variables for multivariable logistic regression

model_df["no_show"] = model_df["no_show"].astype(int)

X = model_df[
    [
        "lead_time_days",
        "appointment_type",
        "booking_channel",
        "hospital_id"
    ]
].copy()

y = model_df["no_show"]

print("Predictor columns:")
print(X.columns.tolist())

print("\nTarget distribution:")
print(y.value_counts())

print("\nTarget rate:")
print(f"No-show rate: {y.mean() * 100:.2f}%")

Predictor columns:
['lead_time_days', 'appointment_type', 'booking_channel', 'hospital_id']

Target distribution:
no_show
0    415965
1     45266
Name: count, dtype: int64

Target rate:
No-show rate: 9.81%


In [31]:
# Encode categorical predictors for logistic regression

X_encoded = pd.get_dummies(
    X,
    columns=[
        "appointment_type",
        "booking_channel",
        "hospital_id"
    ],
    drop_first=True,
    dtype=int
)

print("Encoded predictor columns:")
print(X_encoded.columns.tolist())

print("\nShape of encoded dataset:")
print(X_encoded.shape)

print("\nFirst 5 rows:")
display(X_encoded.head())

Encoded predictor columns:
['lead_time_days', 'appointment_type_Diagnostic', 'appointment_type_Follow-up', 'booking_channel_Phone', 'booking_channel_Referral', 'booking_channel_Walk-in', 'hospital_id_H002', 'hospital_id_H003', 'hospital_id_H004', 'hospital_id_H005', 'hospital_id_H006', 'hospital_id_H007', 'hospital_id_H008']

Shape of encoded dataset:
(461231, 13)

First 5 rows:


,lead_time_days,appointment_type_Diagnostic,appointment_type_Follow-up,booking_channel_Phone,booking_channel_Referral,booking_channel_Walk-in,hospital_id_H002,hospital_id_H003,hospital_id_H004,hospital_id_H005,hospital_id_H006,hospital_id_H007,hospital_id_H008
0,7,0,0,0,1,0,1,0,0,0,0,0,0
1,15,1,0,0,1,0,0,0,0,0,1,0,0
2,10,0,0,0,0,0,0,0,0,0,1,0,0
3,2,0,0,0,0,0,0,0,0,0,0,0,0
4,5,0,1,1,0,0,0,0,1,0,0,0,0


In [32]:
import statsmodels.api as sm

# Add intercept
X_model = sm.add_constant(X_encoded)

# Fit multivariable logistic regression
logit_model = sm.Logit(y, X_model).fit()

print(logit_model.summary())

Optimization terminated successfully.
         Current function value: 0.318473
         Iterations 6
                           Logit Regression Results                           
Dep. Variable:                no_show   No. Observations:               461231
Model:                          Logit   Df Residuals:                   461217
Method:                           MLE   Df Model:                           13
Date:                Sat, 12 Sep 2026   Pseudo R-squ.:                0.007813
Time:                        16:49:50   Log-Likelihood:            -1.4689e+05
converged:                       True   LL-Null:                   -1.4805e+05
Covariance Type:            nonrobust   LLR p-value:                     0.000
                                  coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------------------
const                          -2.3234      0.017   -137.531      0.000   

In [33]:
# Convert logistic regression coefficients into Odds Ratios 

odds_ratios = pd.DataFrame({
    "Odds_Ratio": logit_model.params,
    "P_Value": logit_model.pvalues,
    "CI_Lower": logit_model.conf_int()[0],
    "CI_Upper": logit_model.conf_int()[1]
})

odds_ratios["Odds_Ratio"] = odds_ratios["Odds_Ratio"].apply(lambda x: __import__("math").exp(x))
odds_ratios["CI_Lower"] = odds_ratios["CI_Lower"].apply(lambda x: __import__("math").exp(x))
odds_ratios["CI_Upper"] = odds_ratios["CI_Upper"].apply(lambda x: __import__("math").exp(x))

display(odds_ratios.round(4))

,Odds_Ratio,P_Value,CI_Lower,CI_Upper
const,0.0979,0.0000,0.0947,0.1012
lead_time_days,1.0229,0.0000,1.0216,1.0243
appointment_type_Diagnostic,0.8894,0.0000,0.8635,0.9162
appointment_type_Follow-up,1.2322,0.0000,1.2051,1.2599
booking_channel_Phone,0.7930,0.0000,0.7736,0.8128
booking_channel_Referral,0.6977,0.0000,0.6764,0.7197
booking_channel_Walk-in,0.7795,0.0000,0.7590,0.8005
hospital_id_H002,0.9867,0.4751,0.9512,1.0236
hospital_id_H003,0.9869,0.5191,0.9482,1.0272
hospital_id_H004,0.9869,0.4988,0.9499,1.0254
